<a href="https://colab.research.google.com/github/chelsajn/novapay-funnel_analysis/blob/main/novapay_funnel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np, pandas as pd, sqlite3
rng = np.random.default_rng(42)
n = 10000

users = pd.DataFrame({
    "user_id": range(1, n + 1),
    "signup_time": pd.to_datetime("2026-01-01") + pd.to_timedelta(rng.integers(0, 180*24*60, n), unit="m"),
    "channel": rng.choice(["Organic", "Paid Social", "Referral", "Google Ads"], n, p=[0.30, 0.30, 0.15, 0.25]),
    "device": rng.choice(["Android", "iOS"], n, p=[0.68, 0.32]),
    "city": rng.choice(["Bangalore", "Mumbai", "Delhi", "Hyderabad", "Chennai", "Kochi"], n, p=[0.22, 0.22, 0.20, 0.14, 0.12, 0.10]),
    "age_band": rng.choice(["18-24", "25-34", "35-44", "45+"], n, p=[0.30, 0.40, 0.20, 0.10]),
})

# conversion probabilities (the hidden story)
p_kyc = np.where(users.device == "iOS", 0.82, 0.70)
p_kyc = np.where((users.device == "Android") & (users.channel == "Paid Social"), 0.42, p_kyc)
p_fund = np.where(users.channel == "Referral", 0.85, 0.72)
p_txn = np.where(users.age_band == "45+", 0.70, 0.86)
p_rep = np.where(users.channel == "Referral", 0.68, 0.50)

kyc = rng.random(n) < p_kyc
fund = kyc & (rng.random(n) < p_fund)
txn = fund & (rng.random(n) < p_txn)
rep = txn & (rng.random(n) < p_rep)

def delay(mean_hours):
    return pd.to_timedelta(rng.exponential(mean_hours, n), unit="h")

t_kyc = users.signup_time + delay(6)
t_fund = t_kyc + delay(20)
t_txn = t_fund + delay(10)
t_rep = t_txn + delay(24 * 9)

def stage(name, mask, times):
    return pd.DataFrame({"user_id": users.user_id[mask], "event": name, "event_time": times[mask]})

events = pd.concat([
    stage("signup", np.ones(n, dtype=bool), users.signup_time),
    stage("kyc_verified", kyc, t_kyc),
    stage("wallet_funded", fund, t_fund),
    stage("first_txn", txn, t_txn),
    stage("repeat_txn", rep, t_rep),
]).sort_values(["user_id", "event_time"]).reset_index(drop=True)

# save as CSV (for Power BI) and SQLite (for SQL practice)
users.to_csv("users.csv", index=False)
events.to_csv("events.csv", index=False)
con = sqlite3.connect("novapay.db")
users.to_sql("users", con, if_exists="replace", index=False)
events.to_sql("events", con, if_exists="replace", index=False)

print(events.event.value_counts())

event
signup           10000
kyc_verified      6861
wallet_funded     5050
first_txn         4241
repeat_txn        2249
Name: count, dtype: int64


In [2]:
import sqlite3, pandas as pd
con = sqlite3.connect("novapay.db")
def q(sql): return pd.read_sql(sql, con)

con.execute("DROP VIEW IF EXISTS funnel")
con.execute("""
CREATE VIEW funnel AS
SELECT u.user_id, u.channel, u.device, u.city, u.age_band,
  MIN(CASE WHEN e.event = 'signup' THEN e.event_time END) AS signup_at,
  MIN(CASE WHEN e.event = 'kyc_verified' THEN e.event_time END) AS kyc_at,
  MIN(CASE WHEN e.event = 'wallet_funded' THEN e.event_time END) AS funded_at,
  MIN(CASE WHEN e.event = 'first_txn' THEN e.event_time END) AS txn_at,
  MIN(CASE WHEN e.event = 'repeat_txn' THEN e.event_time END) AS rep_at
FROM users u
LEFT JOIN events e ON u.user_id = e.user_id
GROUP BY u.user_id
""")

In [3]:
q("""
SELECT COUNT(signup_at) AS signups, COUNT(kyc_at) AS kyc, COUNT(funded_at) AS funded,
  COUNT(txn_at) AS first_txn, COUNT(rep_at) AS repeat_txn,
  ROUND(100.0 * COUNT(kyc_at) / COUNT(signup_at), 1) AS signup_to_kyc,
  ROUND(100.0 * COUNT(funded_at) / COUNT(kyc_at), 1) AS kyc_to_fund,
  ROUND(100.0 * COUNT(txn_at) / COUNT(funded_at), 1) AS fund_to_txn,
  ROUND(100.0 * COUNT(rep_at) / COUNT(txn_at), 1) AS txn_to_repeat
FROM funnel
""")

,signups,kyc,funded,first_txn,repeat_txn,signup_to_kyc,kyc_to_fund,fund_to_txn,txn_to_repeat
0,10000,6861,5050,4241,2249,68.6,73.6,84.0,53.0


In [4]:
q("""
SELECT device, channel, COUNT(*) AS users,
  ROUND(100.0 * COUNT(kyc_at) / COUNT(*), 1) AS kyc_rate
FROM funnel
GROUP BY device, channel
ORDER BY kyc_rate
""")

,device,channel,users,kyc_rate
0,Android,Paid Social,2025,42.7
1,Android,Organic,1991,70.1
2,Android,Google Ads,1645,70.3
3,Android,Referral,1061,71.5
4,iOS,Referral,538,79.7
5,iOS,Organic,994,81.8
6,iOS,Paid Social,948,81.9
7,iOS,Google Ads,798,83.8


In [5]:
q("""
SELECT age_band, COUNT(funded_at) AS funded,
  ROUND(100.0 * COUNT(txn_at) / COUNT(funded_at), 1) AS fund_to_txn
FROM funnel
GROUP BY age_band
ORDER BY fund_to_txn
""")

,age_band,funded,fund_to_txn
0,45+,513,69.4
1,18-24,1479,84.9
2,35-44,1062,85.2
3,25-34,1996,86.4


In [6]:
q("""
SELECT channel, COUNT(*) AS users,
  ROUND(100.0 * COUNT(funded_at) / COUNT(kyc_at), 1) AS kyc_to_fund,
  ROUND(100.0 * COUNT(rep_at) / COUNT(txn_at), 1) AS txn_to_repeat
FROM funnel
GROUP BY channel
ORDER BY txn_to_repeat DESC
""")

,channel,users,kyc_to_fund,txn_to_repeat
0,Referral,1599,83.4,65.5
1,Paid Social,2973,72.5,51.6
2,Organic,2985,70.4,50.1
3,Google Ads,2443,72.1,48.3
